# Gold `dim_year`

One row per year (2019–2024). It tells the views which years are the baseline, the recovery years and the comparison window, so those rules are not hardcoded in each view.

| Column | Meaning |
| --- | --- |
| `is_baseline_year` | 2019 (DL-004) |
| `is_recovery_year` | 2022 and 2023 (DL-004) |
| `in_comparison_window` | 2019–2023 (DL-004) |
| `has_dot` | At least one reporting unit has a non-NULL `total_travelers` in that year |
| `has_psa_economy` | At least one reporting unit has a non-NULL value for any of the five economy measures in that year |
| `has_population` | At least one reporting unit has a non-NULL `population_total` in that year |
| `has_all_sources` | `has_dot`, `has_psa_economy` and `has_population` are all true |

**Year-level only.** The `has_*` flags say that a source has usable data for the year. They do **not** say that every province has it. `has_all_sources = true` does not guarantee complete data for every province or for every economic measure. Silver builds a full unit-year grid, so a row can exist with a NULL value. The views must check the fields they need for each province and year.

**Key:** `year`.

**Run log:** the run is `STARTED` first and is marked `SUCCESS` or `SUCCESS_WITH_REVIEW` only after the build, checks, write **and** column comments all finish. Any error marks this `run_id` as `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

## Configuration, helpers and run log

`dq(...)` records each check. `update_run_log(...)` and `save_dq_results()` are separate helpers so the error handler in Step 4 can use them. The row and table counts in the run log come from what was actually written.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from types import SimpleNamespace
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "02_gold_dim_year"
run_id = f"gold_dim_year_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

TBL_DIM_YEAR = "dim_year"
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
BASELINE_YEAR = 2019                 # DL-004
RECOVERY_YEARS = [2022, 2023]        # DL-004
COMPARISON_WINDOW = (2019, 2023)     # DL-004
lo, hi = COMPARISON_WINDOW

DOT_TABLE, ECON_TABLE, POP_TABLE = "dot_travelers_unit_year", "psa_economy_unit_year", "population_unit_year"
ECON_COLS = ["gdp_total_current", "gdp_total_constant", "gdp_per_capita_current", "afs_gva_current", "afs_gva_constant"]

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

# ── Run state ────────────────────────────────────────────────────
dq_results = []
written = {}          # Gold tables fully written in this run: name -> rows
dq_saved = False

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def dq_counts():
    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]
    return blocks, reviews

def update_run_log(status, notes):
    """Update this run_id's row. Table and row counts come from what was actually written so far."""
    blocks, reviews = dq_counts()
    spark.sql(f"""
    UPDATE {gold_table('gold_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": len(written), "rw": sum(written.values()), "dc": len(dq_results),
               "db": len(blocks), "dr": len(reviews), "notes": notes[:2000], "run_id": run_id})

def save_dq_results():
    """Append this run's check results to dq_results once, even when the run stops early."""
    global dq_saved
    if dq_saved or not dq_results:
        return
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))
    dq_saved = True

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

## Step 1: Build

A year counts for a source only if at least one unit has a **non-NULL** value there. A Silver row with only NULLs does not count, because Silver builds a full unit-year grid. The window flags come from the DL-004 constants.

In [0]:
def years_with_data(table, cols):
    """Years in which at least one unit has a non-NULL value in any of `cols`."""
    has_value = reduce(lambda a, b: a | b, [F.col(c).isNotNull() for c in cols])
    return {r[0] for r in spark.table(silver_table(table)).filter(has_value).select("year").distinct().collect()}

def build_dim_year():
    dot_years = years_with_data(DOT_TABLE, ["total_travelers"])
    econ_years = years_with_data(ECON_TABLE, ECON_COLS)
    pop_years = years_with_data(POP_TABLE, ["population_total"])
    dim_year = spark.createDataFrame(
        [(y, y == BASELINE_YEAR, y in RECOVERY_YEARS, lo <= y <= hi,
          (y in dot_years) and (y in econ_years) and (y in pop_years),
          y in dot_years, y in econ_years, y in pop_years) for y in YEARS],
        "year INT, is_baseline_year BOOLEAN, is_recovery_year BOOLEAN, in_comparison_window BOOLEAN, "
        "has_all_sources BOOLEAN, has_dot BOOLEAN, has_psa_economy BOOLEAN, has_population BOOLEAN")
    display(dim_year.orderBy("year"))
    return dim_year, SimpleNamespace(dot_years=dot_years, econ_years=econ_years, pop_years=pop_years)

## Step 2: Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-02 | 6 years, 1 baseline, window 2019–2023, recovery 2022–2023 | BLOCK |
| DQ-GLD-02b | `year` is unique | BLOCK |
| DQ-GLD-02c | `has_all_sources` years equal the years in all three Silver tables; a failure shows both lists | BLOCK |
| DQ-GLD-02d | Every year in 2019–2023 has all three sources; a failure names the missing years | BLOCK |
| DQ-GLD-02e | `year` is an integer column in all three Silver tables | BLOCK |

`DQ-GLD-02c` compares the flags with the Silver sets they were built from, so it mainly guards the build logic. `DQ-GLD-02d` and `02e` are what catch a year that Silver lost or stored as text.

In [0]:
def run_checks(dim_year, sets):
    n = dim_year.count()
    base = dim_year.filter("is_baseline_year").count()
    win = sorted(r[0] for r in dim_year.filter("in_comparison_window").select("year").collect())
    rec = sorted(r[0] for r in dim_year.filter("is_recovery_year").select("year").collect())
    bad = (0 if n == len(YEARS) else 1) + (0 if base == 1 else 1) \
        + (0 if win == list(range(lo, hi + 1)) else 1) + (0 if rec == RECOVERY_YEARS else 1)
    dq("DQ-GLD-02", TBL_DIM_YEAR, "BLOCK", bad,
       f"{len(YEARS)} years, 1 baseline, window {lo}-{hi}, recovery {RECOVERY_YEARS}",
       f"{n} years, {base} baseline, window {win}, recovery {rec}")

    dup = dim_year.groupBy("year").count().filter("count > 1").count()
    dq("DQ-GLD-02b", TBL_DIM_YEAR, "BLOCK", dup, "year is unique", f"{dup} duplicates")

    all_src = sorted(r[0] for r in dim_year.filter("has_all_sources").select("year").collect())
    expected_all = sorted(sets.dot_years & sets.econ_years & sets.pop_years & set(YEARS))
    dq("DQ-GLD-02c", TBL_DIM_YEAR, "BLOCK", 0 if all_src == expected_all else 1,
       "has_all_sources = years in all three Silver tables",
       f"flags {all_src}, Silver {expected_all}")

    gap = sorted(y for y in range(lo, hi + 1) if y not in all_src)
    dq("DQ-GLD-02d", TBL_DIM_YEAR, "BLOCK", len(gap),
       f"every year in {lo}-{hi} has all three sources", f"missing: {gap}", gap,
       notes="Year-level only. Views still check each province's own fields.")

    year_types = {t: dict(spark.table(silver_table(t)).dtypes)["year"] for t in (DOT_TABLE, ECON_TABLE, POP_TABLE)}
    bad_types = {t: ty for t, ty in year_types.items() if ty not in ("int", "bigint", "smallint", "tinyint")}
    dq("DQ-GLD-02e", TBL_DIM_YEAR, "BLOCK", len(bad_types), "year is an integer column in every Silver table",
       f"{year_types}", [f"{t}: {ty}" for t, ty in bad_types.items()])

## Step 3: Write and add column comments

The comments record that the `has_*` flags are year-level. The run is not marked successful until they are done.

In [0]:
def write_dim_year(dim_year):
    (dim_year.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table(TBL_DIM_YEAR)))
    written[TBL_DIM_YEAR] = spark.table(gold_table(TBL_DIM_YEAR)).count()
    print(f"✓ {gold_table(TBL_DIM_YEAR)}: {written[TBL_DIM_YEAR]} rows")

def add_column_comments():
    DY = gold_table(TBL_DIM_YEAR)
    spark.sql(f"COMMENT ON TABLE {DY} IS 'One row per year. has_* flags are year-level availability, not province-level completeness.'")
    comments = {
        "is_baseline_year": "2019, the baseline for recovery and growth (DL-004)",
        "is_recovery_year": "2022 and 2023 (DL-004)",
        "in_comparison_window": "2019 to 2023 (DL-004)",
        "has_dot": "Year-level: at least one unit has non-NULL total_travelers. Not every province.",
        "has_psa_economy": "Year-level: at least one unit has a non-NULL economy measure. Not every province or measure.",
        "has_population": "Year-level: at least one unit has non-NULL population_total. Not every province.",
        "has_all_sources": "True when has_dot, has_psa_economy and has_population are all true. Does not guarantee complete data per province or measure; views must check their own fields.",
    }
    for col, text in comments.items():
        spark.sql(f"ALTER TABLE {DY} ALTER COLUMN `{col}` COMMENT '{text}'")
    print("✓ Column comments added.")

## Step 4: Run

STARTED → build → checks → write → column comments → SUCCESS. Any error marks this run `FAILED`, records which tables were already written, saves the error text, and re-raises the original error.

In [0]:
try:
    dim_year, sets = build_dim_year()
    run_checks(dim_year, sets)
    save_dq_results()

    blocks, _ = dq_counts()
    if blocks:
        raise AssertionError("BLOCK checks failed: " + ", ".join(r[2] for r in blocks)
                             + f". See {dq_table('dq_results')} for run_id = '{run_id}'.")

    write_dim_year(dim_year)
    add_column_comments()

    _, reviews = dq_counts()
    final_status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update_run_log(final_status, notes)
    print(f"Run {run_id}: {final_status}\n{notes}")

except Exception as e:
    after = (f"Tables already written: {', '.join(f'{k}={v} rows' for k, v in written.items())}."
             if written else "No Gold tables written.")
    try:
        save_dq_results()                                   # keep any checks that already ran
    except Exception as dq_err:
        print(f"Could not save DQ results: {dq_err}")
        after += f" DQ results not saved ({type(dq_err).__name__})."
    try:
        update_run_log("FAILED", f"{type(e).__name__}: {e} {after}")   # always attempted, even if the save above failed
    except Exception as log_err:
        print(f"Could not update the run log: {log_err}")
    raise                                                   # re-raises the original error